# Pipeline Lengkap: Training IndoBERT v5 + Validasi Manual (Akurasi Terkoreksi) -- Dataset App Store

Notebook ini menggabungkan dua tahap yang sebelumnya terpisah jadi satu alur kerja utuh:

- **BAGIAN A -- Training IndoBERT v5**: melatih model sentiment analysis di atas dataset App Store hasil scraping, dengan arsitektur dan hyperparameter identik dengan pipeline v5 (dataset Play Store sebelumnya).
- **BAGIAN B -- Validasi Manual & Akurasi Terkoreksi**: mengoreksi potensi self-distillation bias dari proses pelabelan otomatis (mdhugol), dengan membandingkan label otomatis terhadap label manusia pada sampel acak.
- **BAGIAN C -- Ringkasan Gabungan**: membandingkan performa model (Bagian A) dengan kualitas pelabelan (Bagian B), untuk menyimpulkan apakah performa tinggi model benar-benar mencerminkan pemahaman sentimen, atau sebagian berasal dari bias pelabelan yang terwariskan.

**Kenapa dua bagian ini perlu digabung dalam satu narasi:** akurasi model yang dilaporkan (Bagian A) dihitung terhadap data test yang labelnya berasal dari mdhugol -- BUKAN dari manusia. Kalau akurasi model itu jauh lebih tinggi dari akurasi terkoreksi (Bagian B), itu sinyal bahwa sebagian dari performa "tinggi" model sebenarnya adalah model yang berhasil meniru bias mdhugol, bukan murni memahami sentimen. Inilah kenapa laporan skripsi harus mencantumkan KEDUA angka ini, bukan cuma akurasi model saja.

---
# BAGIAN A -- Training Model IndoBERT v5

Mereplikasi persis pipeline `Enhanced_Model_v5_MultiKategori_Beranotasi.ipynb` (arsitektur, hyperparameter, preprocessing), dijalankan di atas dataset App Store (`ulasan_appstore_relabeled.csv`).

**Yang identik dengan v5:** kamus slang & emoji, fungsi `clean_text()`, arsitektur IndoBERT(768D) -> Dropout(0,2) -> Linear(256) -> GELU -> Dropout(0,1) -> Linear(2), differential learning rate (backbone 1e-5, head 1e-4), class weights `balanced` (bukan SMOTE), label smoothing 0.1, split 80/10/10, early stopping berbasis F1 validasi (patience=3, max 7 epoch).

**Yang beda:** sumber data (App Store) dan volume data yang kemungkinan lebih kecil dari 50rb -- ini sudah diekspektasikan.

## Cell 1 — Instalasi Dependencies

In [1]:
!pip install transformers -q

## Cell 2 — Import Library

In [2]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


## Cell 3 — Load Dataset Hasil Relabeling

In [3]:
DATA_PATH = "ulasan_appstore_relabeled.csv"
TEXT_COL  = "ulasan"
LABEL_COL = "label"
KATEGORI_COL = "kategori"

df = pd.read_csv(DATA_PATH)
print("=" * 60)
print("DATASET: App Store Multi-Kategori Relabeled")
print("=" * 60)
print(f"Original Shape: {df.shape}")

df = df[df[LABEL_COL].isin(["positif", "negatif"])].copy()
print(f"Shape setelah filter netral: {df.shape}")

print(f"\nDistribusi label (text-based):")
print(df[LABEL_COL].value_counts())
print(f"\nLabel asli (rating-based) -- hanya referensi:")
print(df["label_original"].value_counts())
print(f"\nDistribusi per kategori:")
print(df[KATEGORI_COL].value_counts())

DATASET: App Store Multi-Kategori Relabeled
Original Shape: (9066, 9)
Shape setelah filter netral: (9066, 9)

Distribusi label (text-based):
label
negatif    6413
positif    2653
Name: count, dtype: int64

Label asli (rating-based) -- hanya referensi:
label_original
negatif    5761
positif    2834
netral      471
Name: count, dtype: int64

Distribusi per kategori:
kategori
ecommerce              1261
dompet_digital         1221
perbankan              1183
transportasi_travel    1114
hiburan_streaming       867
kesehatan_medis         810
pendidikan              797
sosial                  629
komunikasi              442
food_groceries          392
produktivitas           349
Name: count, dtype: int64


## Cell 4 — Preprocessing Teks (Emoji Mapping, Slang Normalization, Dedup)

Disalin persis dari pipeline IndoBERT v5 — kamus slang, pemetaan emoji, fungsi `clean_text()`, dan `drop_duplicates()`. Untuk IndoBERT, stemming dan stopword removal SENGAJA tidak diterapkan karena berisiko merusak konteks semantik model transformer.

In [4]:
SLANG_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak",
    "ngga": "tidak", "tdk": "tidak", "engga": "tidak", "enggak": "tidak",
    "kagak": "tidak", "kaga": "tidak", "ndak": "tidak", "nda": "tidak",
    "bgt": "banget", "bgtt": "banget", "bngt": "banget",
    "skl": "sekali",
    "apk": "aplikasi", "app": "aplikasi", "apps": "aplikasi",
    "dr": "dari", "drpd": "daripada", "dgn": "dengan", "dg": "dengan",
    "sm": "sama", "brsm": "bersama", "pd": "pada", "utk": "untuk",
    "tuk": "untuk", "buat": "untuk",
    "krn": "karena", "karna": "karena", "krna": "karena",
    "yg": "yang", "tp": "tapi", "tpi": "tapi", "ttp": "tetap",
    "ttpi": "tetapi", "ttg": "tentang",
    "sy": "saya", "gw": "saya", "gue": "saya",
    "km": "kamu", "lo": "kamu", "lu": "kamu",
    "udah": "sudah", "udh": "sudah", "dah": "sudah", "sdh": "sudah",
    "blm": "belum", "blum": "belum",
    "skrg": "sekarang", "skrng": "sekarang",
    "lg": "lagi", "lgi": "lagi",
    "trs": "terus", "trus": "terus",
    "msh": "masih", "masi": "masih",
    "hbs": "habis",
    "aja": "saja", "aj": "saja",
    "emg": "memang", "emang": "memang",
    "nih": "ini", "tuh": "itu",
    "bkn": "bukan",
    "klo": "kalau", "klu": "kalau", "kl": "kalau", "klau": "kalau",
    "gimana": "bagaimana", "gmn": "bagaimana",
    "knp": "kenapa",
    "lbh": "lebih",
    "kyk": "seperti", "kyak": "seperti", "kayak": "seperti",
    "jd": "jadi", "jdi": "jadi",
    "bs": "bisa", "bsa": "bisa",
    "hrs": "harus",
    "jg": "juga",
    "mo": "mau",
    "dpt": "dapat", "dpat": "dapat",
    "sdkt": "sedikit",
    "sampe": "sampai", "ampe": "sampai",
    "bentar": "sebentar",
    "pake": "pakai", "pk": "pakai",
    "nunggu": "menunggu",
    "nyari": "mencari",
    "tmn": "teman",
    "ok": "oke",
    "mantap": "bagus", "mantul": "bagus", "kece": "bagus",
}

EMOJI_MAP = {
    "\U0001F44D": " bagus ", "\U0001F44E": " jelek ", "\U0001F621": " kecewa ", "\U0001F62D": " sedih ",
    "\U0001F60A": " senang ", "\U0001F60D": " suka ", "\u2764\ufe0f": " cinta ", "\U0001F31F": " mantap ",
    "\U0001F622": " sedih ", "\U0001F620": " marah ", "\U0001F612": " kesal ", "\U0001F618": " suka ",
    "\U0001F601": " senang ", "\U0001F44C": " oke ", "\U0001F44F": " bagus ", "\U0001F496": " cinta ",
    "\U0001F389": " senang ", "\U0001F610": " biasa ", "\U0001F914": " biasa ", "\U0001F937": " biasa ",
}

def replace_emojis(text):
    for emo, txt in EMOJI_MAP.items():
        text = text.replace(emo, txt)
    return text

def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def replace_slang(text):
    tokens = text.split()
    return " ".join([SLANG_MAP.get(tok, tok) for tok in tokens])

def clean_text(text):
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    text = replace_emojis(text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\w\s.,!?;:'\"()\-/%]", " ", text, flags=re.UNICODE)
    text = normalize_repeated_chars(text)
    text = replace_slang(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df[TEXT_COL] = df[TEXT_COL].apply(clean_text)
df = df[df[TEXT_COL].str.len() > 0]
df = df.drop_duplicates(subset=TEXT_COL).reset_index(drop=True)
print(f"Shape setelah cleaning + dedup: {df.shape}")
df[[TEXT_COL, LABEL_COL, KATEGORI_COL]].head(5)

Shape setelah cleaning + dedup: (8859, 9)


,ulasan,label,kategori
0,mantal,positif,ecommerce
1,tiap abis lama tidak pakai tokped pasti bug ba...,negatif,ecommerce
2,makasih,positif,ecommerce
3,pembayaran menggunakan saldo tokopedia yang te...,positif,ecommerce
4,pesan instan tapi proses pickup lebih dari seh...,negatif,ecommerce


## Cell 5 — Encoding Label & Split Dataset (80/10/10)

Label diubah jadi angka (`negatif`=0, `positif`=1). Split dilakukan stratified (proporsi kelas dijaga sama di train/val/test) sesuai Bab 3.3.4 skripsi.

In [5]:
LABEL2ID = {"negatif": 0, "positif": 1}
ID2LABEL = {0: "negatif", 1: "positif"}

df["label_id"] = df[LABEL_COL].map(LABEL2ID)

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label_id"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train : {train_df.shape}  | dist: {train_df['label_id'].value_counts().sort_index().to_dict()}")
print(f"Val   : {val_df.shape}   | dist: {val_df['label_id'].value_counts().sort_index().to_dict()}")
print(f"Test  : {test_df.shape}   | dist: {test_df['label_id'].value_counts().sort_index().to_dict()}")

print("\n--- Distribusi kategori di data Test ---")
print(test_df[KATEGORI_COL].value_counts())

Train : (7087, 10)  | dist: {0: 5093, 1: 1994}
Val   : (886, 10)   | dist: {0: 637, 1: 249}
Test  : (886, 10)   | dist: {0: 636, 1: 250}

--- Distribusi kategori di data Test ---
kategori
transportasi_travel    128
dompet_digital         125
perbankan              115
ecommerce              105
hiburan_streaming       87
kesehatan_medis         79
pendidikan              72
sosial                  68
food_groceries          42
komunikasi              40
produktivitas           25
Name: count, dtype: int64


## Cell 6 — Hitung Class Weights

`class_weight='balanced'` (BUKAN SMOTE), konsisten dengan v5, supaya loss function memberi bobot lebih besar ke kelas minoritas.

In [6]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label_id"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Class weights: {class_weights_tensor}")
print(f"  negatif (0): {class_weights[0]:.4f}")
print(f"  positif (1): {class_weights[1]:.4f}")

Class weights: tensor([0.6958, 1.7771], device='cuda:0')
  negatif (0): 0.6958
  positif (1): 1.7771


## Cell 7 — Load Tokenizer IndoBERT

In [7]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer '{MODEL_NAME}' berhasil dimuat. Max length: {MAX_LENGTH}")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer 'indobenchmark/indobert-base-p1' berhasil dimuat. Max length: 256


## Cell 8 — Dataset Class

In [8]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = SentimentDataset(train_df[TEXT_COL].tolist(), train_df["label_id"].tolist(), tokenizer, MAX_LENGTH)
val_dataset   = SentimentDataset(val_df[TEXT_COL].tolist(),   val_df["label_id"].tolist(),   tokenizer, MAX_LENGTH)
test_dataset  = SentimentDataset(test_df[TEXT_COL].tolist(),  test_df["label_id"].tolist(),  tokenizer, MAX_LENGTH)
print(f"Dataset sizes -- Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset sizes -- Train: 7087, Val: 886, Test: 886


## Cell 9 — Arsitektur Model (Identik dengan v5)

Backbone IndoBERT (768D), ambil representasi token `[CLS]`, teruskan ke classifier head: Dropout(0,2) → Linear(256) → GELU → Dropout(0,1) → Linear(2). Loss `CrossEntropyLoss` dengan `class_weights` dan `label_smoothing=0.1`.

In [9]:
class IndoBERTEnhanced(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits     = self.classifier(cls_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=class_weights_tensor,
                label_smoothing=0.1,
            )
            loss = loss_fct(logits, labels)

        return loss, logits

model = IndoBERTEnhanced(MODEL_NAME, num_labels=2, dropout=0.2)
model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Total params    : 124,638,722
Trainable params: 124,638,722


## Cell 10 — Hyperparameter, DataLoader, Optimizer & Scheduler

Batch size 16 dengan gradient accumulation 2x (effective batch size 32), max 7 epoch. Differential learning rate: backbone IndoBERT (1e-5), classifier head baru (1e-4). Scheduler: warmup linear di awal, lalu turun linear.

In [10]:
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
MAX_EPOCHS = 7
PATIENCE = 3
LR_BACKBONE = 1e-5
LR_HEAD = 1e-4
WARMUP_RATIO = 0.1

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer_params = [
    {"params": model.bert.parameters(), "lr": LR_BACKBONE},
    {"params": model.classifier.parameters(), "lr": LR_HEAD},
]
optimizer = torch.optim.AdamW(optimizer_params, weight_decay=0.01)

total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f"Total training steps : {total_steps}")
print(f"Warmup steps          : {warmup_steps}")
print(f"Effective batch size  : {BATCH_SIZE * GRAD_ACCUM_STEPS}")

Total training steps : 1547
Warmup steps          : 154
Effective batch size  : 32


## Cell 11 — Fungsi Evaluasi (Validasi/Test)

In [11]:
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            loss, logits = model(input_ids, attention_mask, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    avg_loss = total_loss / len(dataloader)

    return {
        "loss": avg_loss, "accuracy": acc,
        "precision": prec, "recall": rec, "f1": f1,
        "preds": all_preds, "labels": all_labels,
    }

print("Fungsi evaluasi siap.")

Fungsi evaluasi siap.


## Cell 12 — Training Loop dengan Early Stopping

Model dievaluasi tiap akhir epoch di data validasi. Kalau F1 validasi membaik, bobot model disimpan sebagai "model terbaik". Kalau tidak membaik selama `PATIENCE` (3) epoch berturut-turut, training dihentikan lebih awal. Bobot terbaik (bukan bobot epoch terakhir) yang dipakai untuk evaluasi final.

In [12]:
history = []
best_val_f1 = 0.0
epochs_no_improve = 0
best_model_state = None

print("=" * 60)
print("MULAI TRAINING -- IndoBERT v5 (App Store Dataset, Binary Setup)")
print("=" * 60)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        loss, logits = model(input_ids, attention_mask, labels)
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % 20 == 0:
            print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | Step {step:03d}/{len(train_loader)} | Batch Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}")

    avg_train_loss = total_loss / len(train_loader)
    val_metrics = evaluate(model, val_loader)

    print(f"\n>> Epoch {epoch} selesai | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_metrics['loss']:.4f} | Val F1: {val_metrics['f1']:.4f} | "
          f"Val Acc: {val_metrics['accuracy']:.4f}\n")

    history.append({
        "epoch": epoch, "train_loss": avg_train_loss,
        "val_loss": val_metrics["loss"], "val_f1": val_metrics["f1"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        print(f"   -> F1 validasi membaik ({best_val_f1:.4f}), model disimpan.")
    else:
        epochs_no_improve += 1
        print(f"   -> F1 validasi tidak membaik ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping di epoch {epoch}.")
            break

model.load_state_dict(best_model_state)
print(f"\nTraining selesai. Best Val F1: {best_val_f1:.4f}")

MULAI TRAINING -- IndoBERT v5 (App Store Dataset, Binary Setup)
Epoch 01/7 | Step 000/443 | Batch Loss: 0.6918
Epoch 01/7 | Step 020/443 | Batch Loss: 0.6471
Epoch 01/7 | Step 040/443 | Batch Loss: 0.7105
Epoch 01/7 | Step 060/443 | Batch Loss: 0.6388
Epoch 01/7 | Step 080/443 | Batch Loss: 0.5881
Epoch 01/7 | Step 100/443 | Batch Loss: 0.4675
Epoch 01/7 | Step 120/443 | Batch Loss: 0.4356
Epoch 01/7 | Step 140/443 | Batch Loss: 0.2399
Epoch 01/7 | Step 160/443 | Batch Loss: 0.3883
Epoch 01/7 | Step 180/443 | Batch Loss: 0.4699
Epoch 01/7 | Step 200/443 | Batch Loss: 0.3216
Epoch 01/7 | Step 220/443 | Batch Loss: 0.2412
Epoch 01/7 | Step 240/443 | Batch Loss: 0.3709
Epoch 01/7 | Step 260/443 | Batch Loss: 0.4546
Epoch 01/7 | Step 280/443 | Batch Loss: 0.2849
Epoch 01/7 | Step 300/443 | Batch Loss: 0.2826
Epoch 01/7 | Step 320/443 | Batch Loss: 0.2501
Epoch 01/7 | Step 340/443 | Batch Loss: 0.3710
Epoch 01/7 | Step 360/443 | Batch Loss: 0.4765
Epoch 01/7 | Step 380/443 | Batch Loss: 0.2

## Cell 13 — Evaluasi Final di Data Test

In [13]:
test_metrics = evaluate(model, test_loader)
y_true = test_metrics["labels"]
y_pred = test_metrics["preds"]

test_acc  = test_metrics["accuracy"]
test_prec = test_metrics["precision"]
test_rec  = test_metrics["recall"]
test_f1   = test_metrics["f1"]

print("=" * 60)
print("EVALUASI FINAL -- DATA TEST")
print("=" * 60)
print(f"Test Accuracy  (mac) : {test_acc:.4f}")
print(f"Test Precision (mac) : {test_prec:.4f}")
print(f"Test Recall    (mac) : {test_rec:.4f}")
print(f"Test F1        (mac) : {test_f1:.4f}")

print("\n-- Classification Report ----------------------------------------")
print(classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in range(2)], zero_division=0))

print("-- Confusion Matrix ---------------------------------------------")
print(confusion_matrix(y_true, y_pred))

print("\n-- Training History ---------------------------------------------")
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

EVALUASI FINAL -- DATA TEST
Test Accuracy  (mac) : 0.9808
Test Precision (mac) : 0.9747
Test Recall    (mac) : 0.9781
Test F1        (mac) : 0.9764

-- Classification Report ----------------------------------------
              precision    recall  f1-score   support

     negatif       0.99      0.98      0.99       636
     positif       0.96      0.97      0.97       250

    accuracy                           0.98       886
   macro avg       0.97      0.98      0.98       886
weighted avg       0.98      0.98      0.98       886

-- Confusion Matrix ---------------------------------------------
[[626  10]
 [  7 243]]

-- Training History ---------------------------------------------
 epoch  train_loss  val_loss   val_f1  val_accuracy
     1    0.396748  0.289118 0.973236      0.978555
     2    0.280411  0.299925 0.973303      0.978555
     3    0.262929  0.300247 0.973168      0.978555
     4    0.254243  0.302361 0.966319      0.972912
     5    0.251226  0.302865 0.971862     

## Cell 14 — Evaluasi Per Kategori Aplikasi

Performa dihitung terpisah per kategori, supaya kelihatan apakah model konsisten bagus di semua domain aplikasi, atau ada kategori tertentu yang lebih sulit dikenali.

In [14]:
model.eval()
test_df_eval = test_df.copy()
test_df_eval["pred"] = None

with torch.no_grad():
    preds_all = []
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        _, logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        preds_all.extend(preds)

test_df_eval["pred"] = preds_all

hasil_per_kategori = []
for kategori in test_df_eval[KATEGORI_COL].unique():
    subset = test_df_eval[test_df_eval[KATEGORI_COL] == kategori]
    if len(subset) == 0:
        continue
    acc = accuracy_score(subset["label_id"], subset["pred"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        subset["label_id"], subset["pred"], average="macro", zero_division=0
    )
    hasil_per_kategori.append({
        "kategori": kategori, "n_data": len(subset),
        "accuracy": acc, "precision_macro": prec,
        "recall_macro": rec, "f1_macro": f1,
    })

df_hasil_kategori = pd.DataFrame(hasil_per_kategori).sort_values("f1_macro", ascending=False)
print("=" * 70)
print("PERFORMA MODEL PER KATEGORI APLIKASI (App Store Dataset)")
print("=" * 70)
print(df_hasil_kategori.to_string(index=False))

PERFORMA MODEL PER KATEGORI APLIKASI (App Store Dataset)
           kategori  n_data  accuracy  precision_macro  recall_macro  f1_macro
     dompet_digital     125  1.000000         1.000000      1.000000  1.000000
     food_groceries      42  1.000000         1.000000      1.000000  1.000000
          ecommerce     105  0.990476         0.993056      0.985294  0.989041
transportasi_travel     128  0.984375         0.975000      0.988889  0.981562
    kesehatan_medis      79  0.974684         0.979167      0.969697  0.973737
          perbankan     115  0.973913         0.972653      0.964415  0.968409
  hiburan_streaming      87  0.977011         0.959722      0.959722  0.959722
      produktivitas      25  0.960000         0.968750      0.950000  0.957555
         pendidikan      72  0.972222         0.955665      0.955665  0.955665
             sosial      68  0.955882         0.900319      0.932759  0.915528
         komunikasi      40  0.975000         0.986842      0.833333  0.89

## Cell 15 — Simpan Model & Metadata

In [15]:
import os

SAVE_DIR = "sentiment_model_appstore_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model.pt"))
tokenizer.save_pretrained(SAVE_DIR)

metadata = pd.DataFrame([{
    "model_version": "appstore_v1",
    "dataset": DATA_PATH,
    "cakupan": f"{df[KATEGORI_COL].nunique()} kategori aplikasi (App Store)",
    "labeling_method": "text-based (mdhugol) + rating sebagai referensi",
    "class_imbalance_handling": "class weights (balanced) -- BUKAN SMOTE",
    "test_accuracy": test_acc,
    "test_f1_macro": test_f1,
    "test_precision_macro": test_prec,
    "test_recall_macro": test_rec,
    "epochs_trained": len(history),
    "best_val_f1": best_val_f1,
}])
metadata.to_csv(os.path.join(SAVE_DIR, "model_metadata.csv"), index=False)

print(f"Model disimpan di: {SAVE_DIR}/")
print(metadata.T)

Model disimpan di: sentiment_model_appstore_v1/
                                                                        0
model_version                                                 appstore_v1
dataset                                     ulasan_appstore_relabeled.csv
cakupan                                  11 kategori aplikasi (App Store)
labeling_method           text-based (mdhugol) + rating sebagai referensi
class_imbalance_handling          class weights (balanced) -- BUKAN SMOTE
test_accuracy                                                    0.980813
test_f1_macro                                                    0.976403
test_precision_macro                                             0.974708
test_recall_macro                                                0.978138
epochs_trained                                                          5
best_val_f1                                                      0.973303


---
# BAGIAN B -- Validasi Manual & Akurasi Terkoreksi

**Kenapa langkah ini penting (self-distillation bias):** label training Bagian A dihasilkan oleh model pretrained lain (`mdhugol/indonesia-bert-sentiment-classification`), yang basis arsitekturnya juga IndoBERT. Ketika model v5 dilatih untuk "menebak" label dari mdhugol, ada risiko model justru mempelajari pola keputusan mdhugol, bukan sentimen sebenarnya dari teks. Akibatnya, akurasi model terhadap test set (Bagian A) bisa inflated.

**Cara mengoreksinya:** sampel acak (`sample_validasi_manual_appstore.xlsx`) sudah dianotasi manual oleh peneliti (kolom `label_manual`), secara independen tanpa melihat label mdhugol terlebih dahulu (blind annotation). Bagian ini menghitung seberapa sering label mdhugol cocok dengan penilaian manusia -- angka inilah yang mencerminkan kualitas pelabelan yang sesungguhnya, bukan akurasi model terhadap test set berlabel mdhugol semata.

Ini adalah replikasi metodologi yang sama seperti pada dataset Play Store sebelumnya (menghasilkan koreksi dari 97,79% -> 96,40%).

## Cell 16 — Instalasi Dependencies

In [16]:
!pip install pandas scikit-learn openpyxl -q

## Cell 17 — Import Library

In [17]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

print("Library siap.")

Library siap.


## Cell 18 — Load Data Hasil Validasi Manual

File ini adalah hasil dari `sample_validasi_manual_appstore.csv` (dibuat otomatis di notebook relabeling) yang SUDAH diisi kolom `label_manual` oleh peneliti secara manual, tanpa melihat `label_otomatis` terlebih dahulu (blind annotation) supaya penilaian tidak bias oleh label mdhugol.

In [18]:
DATA_PATH = "sample_validasi_manual_appstore.xlsx"  # ganti .csv kalau formatnya csv

if DATA_PATH.endswith(".xlsx"):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f"Total sampel: {len(df)}")
print(f"Kolom tersedia: {df.columns.tolist()}")
df.head()

Total sampel: 242
Kolom tersedia: ['ulasan', 'label_otomatis', 'kategori', 'rating', 'nama_aplikasi', 'label_manual']


,ulasan,label_otomatis,kategori,rating,nama_aplikasi,label_manual
0,"intinya jangan pake ovo, duit gua nyangkut di ...",negatif,dompet_digital,1,OVO,negatif
1,"Apa apa an ini,transfer ga bisaa",negatif,dompet_digital,1,DANA Dompet Digital Indonesia,negatif
2,Siap ada saldo selalu berkurang buat tagihan y...,negatif,dompet_digital,1,DANA Dompet Digital Indonesia,negatif
3,masa kesalahan teknis trs trsan sih?,negatif,dompet_digital,2,"GoPay: Transfer, Bayar, QRIS",negatif
4,"Mahal banget top up kena biaya admin, transfer...",negatif,dompet_digital,1,OVO,negatif


## Cell 19 — Validasi Kelengkapan Data

Sebelum menghitung metrik, pastikan tidak ada baris yang label manualnya masih kosong (belum diisi), dan tidak ada nilai label yang typo/di luar dua kelas yang diharapkan (`negatif`/`positif`).

In [19]:
print("=" * 60)
print("VALIDASI KELENGKAPAN DATA")
print("=" * 60)

missing = df["label_manual"].isna().sum()
print(f"Baris dengan label_manual kosong: {missing}")

if missing > 0:
    print("\n PERINGATAN: ada baris belum dilabeli, baris ini akan di-drop dari perhitungan.")
    df = df.dropna(subset=["label_manual"]).reset_index(drop=True)

nilai_valid = {"positif", "negatif"}
label_manual_aneh = set(df["label_manual"].unique()) - nilai_valid
label_otomatis_aneh = set(df["label_otomatis"].unique()) - nilai_valid

if label_manual_aneh:
    print(f"\n PERINGATAN: ada nilai tidak dikenal di label_manual: {label_manual_aneh}")
    print("   Cek typo (misal 'Positif' dengan huruf besar, spasi tersembunyi, dll)")
if label_otomatis_aneh:
    print(f"\n PERINGATAN: ada nilai tidak dikenal di label_otomatis: {label_otomatis_aneh}")

if not label_manual_aneh and not label_otomatis_aneh:
    print("\nData bersih, semua label sesuai format yang diharapkan (positif/negatif).")

print(f"\nDistribusi label_manual (ground truth):")
print(df["label_manual"].value_counts())
print(f"\nDistribusi label_otomatis (mdhugol):")
print(df["label_otomatis"].value_counts())

VALIDASI KELENGKAPAN DATA
Baris dengan label_manual kosong: 0

Data bersih, semua label sesuai format yang diharapkan (positif/negatif).

Distribusi label_manual (ground truth):
label_manual
negatif    169
positif     73
Name: count, dtype: int64

Distribusi label_otomatis (mdhugol):
label_otomatis
negatif    168
positif     74
Name: count, dtype: int64


## Cell 20 — Hitung Akurasi Terkoreksi

Bandingkan `label_otomatis` (prediksi mdhugol) terhadap `label_manual` (ground truth dari peneliti). Ini bukan mengevaluasi performa model IndoBERT v5, melainkan mengevaluasi **kualitas proses pelabelan** yang jadi basis training model tersebut.

In [20]:
y_true = df["label_manual"]
y_pred = df["label_otomatis"]

akurasi_terkoreksi = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

print("=" * 60)
print("HASIL: AKURASI TERKOREKSI (label_otomatis vs label_manual)")
print("=" * 60)
print(f"Jumlah sampel divalidasi : {len(df)}")
print(f"Akurasi terkoreksi        : {akurasi_terkoreksi*100:.2f}%")
print(f"Precision (macro)         : {prec:.4f}")
print(f"Recall (macro)            : {rec:.4f}")
print(f"F1 (macro)                : {f1:.4f}")

HASIL: AKURASI TERKOREKSI (label_otomatis vs label_manual)
Jumlah sampel divalidasi : 242
Akurasi terkoreksi        : 97.93%
Precision (macro)         : 0.9738
Recall (macro)            : 0.9774
F1 (macro)                : 0.9756


## Cell 21 — Classification Report & Confusion Matrix

Detail performa per kelas (positif/negatif), untuk melihat apakah mdhugol lebih sering salah di satu arah tertentu (misal cenderung terlalu sering memprediksi negatif).

In [21]:
print("-- Classification Report (label_otomatis vs label_manual) --\n")
print(classification_report(y_true, y_pred, zero_division=0))

print("-- Confusion Matrix --")
cm = confusion_matrix(y_true, y_pred, labels=["negatif", "positif"])
cm_df = pd.DataFrame(
    cm,
    index=["true_negatif (manual)", "true_positif (manual)"],
    columns=["pred_negatif (otomatis)", "pred_positif (otomatis)"],
)
print(cm_df)

-- Classification Report (label_otomatis vs label_manual) --

              precision    recall  f1-score   support

     negatif       0.99      0.98      0.99       169
     positif       0.96      0.97      0.97        73

    accuracy                           0.98       242
   macro avg       0.97      0.98      0.98       242
weighted avg       0.98      0.98      0.98       242

-- Confusion Matrix --
                       pred_negatif (otomatis)  pred_positif (otomatis)
true_negatif (manual)                      166                        3
true_positif (manual)                        2                       71


## Cell 22 — Analisis Kasus Ketidaksepakatan

Tampilkan baris-baris di mana mdhugol dan penilaian manusia berbeda. Ini bagian paling berguna untuk Bab IV — dari sini bisa dilihat POLA kesalahan (misal: review panjang & campur bahasa, sarkasme, kata-kata emosional intens, atau teks terlalu pendek/ambigu).

In [22]:
disagree = df[df["label_manual"] != df["label_otomatis"]].copy()

print("=" * 60)
print(f"TOTAL KETIDAKSEPAKATAN: {len(disagree)} dari {len(df)} sampel ({len(disagree)/len(df)*100:.1f}%)")
print("=" * 60)

# Breakdown arah kesalahan
salah_ke_negatif = disagree[disagree["label_otomatis"] == "negatif"]
salah_ke_positif = disagree[disagree["label_otomatis"] == "positif"]
print(f"\nOtomatis bilang NEGATIF, tapi manual bilang POSITIF: {len(salah_ke_negatif)} kasus")
print(f"Otomatis bilang POSITIF, tapi manual bilang NEGATIF: {len(salah_ke_positif)} kasus")

print("\n-- Contoh kasus ketidaksepakatan (10 pertama) --")
for idx, row in disagree.head(10).iterrows():
    teks_singkat = row["ulasan"][:150] + ("..." if len(row["ulasan"]) > 150 else "")
    print(f"\n[{row['kategori']}] {row['nama_aplikasi']}")
    print(f"  Teks     : {teks_singkat}")
    print(f"  Otomatis : {row['label_otomatis']}  |  Manual: {row['label_manual']}")

disagree.to_csv("ketidaksepakatan_label_appstore.csv", index=False)
print(f"\nSemua kasus ketidaksepakatan disimpan: ketidaksepakatan_label_appstore.csv")

TOTAL KETIDAKSEPAKATAN: 5 dari 242 sampel (2.1%)

Otomatis bilang NEGATIF, tapi manual bilang POSITIF: 2 kasus
Otomatis bilang POSITIF, tapi manual bilang NEGATIF: 3 kasus

-- Contoh kasus ketidaksepakatan (10 pertama) --

[food_groceries] Astro - Groceries in Minutes
  Teks     : please buka dibali siapapun
  Otomatis : negatif  |  Manual: positif

[kesehatan_medis] Alodokter: Chat Bersama Dokter
  Teks     : Mending gk usah dowload app ini ,masa iya gue order caladine tanpa resep dokterpun masih ada problem ,dah gue minta refund dana malah dibilang 3 hari ...
  Otomatis : positif  |  Manual: negatif

[produktivitas] Microsoft Office
  Teks     : Skrip
  Otomatis : positif  |  Manual: negatif

[produktivitas] Notion: Catatan, Tugas, AI
  Teks     : Joss pokoke berguna wenak lak maju terus gae notion
  Otomatis : negatif  |  Manual: positif

[transportasi_travel] Grab: Pesan Ojek & Makanan
  Teks     : Kmpang
  Otomatis : positif  |  Manual: negatif

Semua kasus ketidaksepakatan disimp

## Cell 23 — Ringkasan untuk Bab IV

Rangkuman siap pakai untuk ditulis di draft skripsi, termasuk perbandingan dengan temuan self-distillation bias pada dataset Play Store sebelumnya (kalau ingin dibandingkan lintas dataset).

In [23]:
ringkasan_lines = [
    "=" * 70,
    "RINGKASAN UNTUK BAB IV -- VALIDASI MANUAL DATASET APP STORE",
    "=" * 70,
    "",
    f"Jumlah sampel divalidasi manual : {len(df)}",
    "Akurasi label otomatis (mdhugol) dibanding label manual (ground truth):",
    f"  -> {akurasi_terkoreksi*100:.2f}%",
    "",
    f"Precision (macro) : {prec:.4f}",
    f"Recall (macro)    : {rec:.4f}",
    f"F1 (macro)        : {f1:.4f}",
    "",
    "Interpretasi:",
    "Angka ini merepresentasikan akurasi TERKOREKSI dari proses pelabelan,",
    "yang mengoreksi kemungkinan self-distillation bias antara model pelabelan",
    "(mdhugol) dan model yang dilatih (IndoBERT v5) -- keduanya berbasis",
    "arsitektur yang sama, sehingga akurasi model terhadap data test berlabel",
    "mdhugol semata berisiko inflated.",
]

for line in ringkasan_lines:
    print(line)

print()
print("Bandingkan dengan akurasi model IndoBERT v5 terhadap test set (dari notebook training):")
print(f"  -> Jika akurasi model jauh lebih tinggi dari {akurasi_terkoreksi*100:.2f}%,")
print(f"     itu mengindikasikan model turut mewarisi bias pelabelan mdhugol,")
print(f"     bukan murni performa sesungguhnya dalam memahami sentimen.")

RINGKASAN UNTUK BAB IV -- VALIDASI MANUAL DATASET APP STORE

Jumlah sampel divalidasi manual : 242
Akurasi label otomatis (mdhugol) dibanding label manual (ground truth):
  -> 97.93%

Precision (macro) : 0.9738
Recall (macro)    : 0.9774
F1 (macro)        : 0.9756

Interpretasi:
Angka ini merepresentasikan akurasi TERKOREKSI dari proses pelabelan,
yang mengoreksi kemungkinan self-distillation bias antara model pelabelan
(mdhugol) dan model yang dilatih (IndoBERT v5) -- keduanya berbasis
arsitektur yang sama, sehingga akurasi model terhadap data test berlabel
mdhugol semata berisiko inflated.

Bandingkan dengan akurasi model IndoBERT v5 terhadap test set (dari notebook training):
  -> Jika akurasi model jauh lebih tinggi dari 97.93%,
     itu mengindikasikan model turut mewarisi bias pelabelan mdhugol,
     bukan murni performa sesungguhnya dalam memahami sentimen.


---
# BAGIAN C -- Ringkasan Gabungan: Performa Model vs Kualitas Pelabelan

Membandingkan hasil Bagian A (akurasi model IndoBERT v5 di test set) dengan Bagian B (akurasi terkoreksi dari validasi manual). Selisih besar antara keduanya mengindikasikan seberapa besar performa model "dibantu" oleh bias pelabelan otomatis, bukan murni kemampuan model memahami sentimen.

In [24]:
print("=" * 70)
print("RINGKASAN GABUNGAN -- MODEL vs KUALITAS PELABELAN")
print("=" * 70)

print()
print("[Bagian A] Performa Model IndoBERT v5 (test set, label dari mdhugol):")
print(f"  Test Accuracy   : {test_acc*100:.2f}%")
print(f"  Test F1 (macro) : {test_f1*100:.2f}%")

print()
print("[Bagian B] Akurasi Terkoreksi (label mdhugol vs label manusia):")
print(f"  Akurasi terkoreksi : {akurasi_terkoreksi*100:.2f}%")
print(f"  F1 (macro)         : {f1*100:.2f}%")

selisih_acc = (test_acc - akurasi_terkoreksi) * 100
selisih_f1 = (test_f1 - f1) * 100

print()
print("[Perbandingan]")
print(f"  Selisih Accuracy : {selisih_acc:+.2f} poin persen")
print(f"  Selisih F1       : {selisih_f1:+.2f} poin persen")

print()
print("Interpretasi:")
if selisih_acc > 3:
    interpretasi_lines = [
        "  Selisih cukup besar (" + f"{selisih_acc:.2f}" + " poin). Ini mengindikasikan model",
        "  IndoBERT v5 kemungkinan turut mewarisi sebagian bias pelabelan mdhugol",
        "  (self-distillation bias) -- performa tinggi model terhadap test set TIDAK",
        "  sepenuhnya mencerminkan pemahaman sentimen yang sesungguhnya.",
        "  Rekomendasi: laporkan KEDUA angka di Bab IV -- akurasi model (mentah) dan",
        "  akurasi terkoreksi (dari validasi manual) -- sebagai bentuk transparansi",
        "  metodologis, sama seperti perlakuan pada dataset Play Store sebelumnya."
    ]
else:
    interpretasi_lines = [
        "  Selisih relatif kecil (" + f"{selisih_acc:.2f}" + " poin), mengindikasikan performa",
        "  model cukup konsisten dengan kualitas pelabelan dasarnya. Tetap disarankan",
        "  melaporkan kedua angka di Bab IV untuk transparansi metodologis."
    ]

for line in interpretasi_lines:
    print(line)

ringkasan_final = pd.DataFrame([{
    "metrik": "Accuracy",
    "model_terhadap_test_set (Bagian A)": f"{test_acc*100:.2f}%",
    "akurasi_terkoreksi (Bagian B)": f"{akurasi_terkoreksi*100:.2f}%",
    "selisih_poin_persen": f"{selisih_acc:+.2f}",
}, {
    "metrik": "F1 (macro)",
    "model_terhadap_test_set (Bagian A)": f"{test_f1*100:.2f}%",
    "akurasi_terkoreksi (Bagian B)": f"{f1*100:.2f}%",
    "selisih_poin_persen": f"{selisih_f1:+.2f}",
}])
print()
print("=" * 70)
print("TABEL RINGKASAN (siap disalin ke Bab IV)")
print("=" * 70)
print(ringkasan_final.to_string(index=False))

ringkasan_final.to_csv("ringkasan_gabungan_model_vs_validasi_manual.csv", index=False)
print()
print("Disimpan: ringkasan_gabungan_model_vs_validasi_manual.csv")

RINGKASAN GABUNGAN -- MODEL vs KUALITAS PELABELAN

[Bagian A] Performa Model IndoBERT v5 (test set, label dari mdhugol):
  Test Accuracy   : 98.08%
  Test F1 (macro) : 97.64%

[Bagian B] Akurasi Terkoreksi (label mdhugol vs label manusia):
  Akurasi terkoreksi : 97.93%
  F1 (macro)         : 97.56%

[Perbandingan]
  Selisih Accuracy : +0.15 poin persen
  Selisih F1       : +0.08 poin persen

Interpretasi:
  Selisih relatif kecil (0.15 poin), mengindikasikan performa
  model cukup konsisten dengan kualitas pelabelan dasarnya. Tetap disarankan
  melaporkan kedua angka di Bab IV untuk transparansi metodologis.

TABEL RINGKASAN (siap disalin ke Bab IV)
    metrik model_terhadap_test_set (Bagian A) akurasi_terkoreksi (Bagian B) selisih_poin_persen
  Accuracy                             98.08%                        97.93%               +0.15
F1 (macro)                             97.64%                        97.56%               +0.08

Disimpan: ringkasan_gabungan_model_vs_validasi_manual.